# Assignment 2: Transformer language models

Build an OLMo-2-style Transformer from scratch, train on the same Wikipedia data as A1, and generate text. Reuses A1's tokenizer and trainer.

All Transformer components live in `A2_skeleton.py`.

## Setup

In [ ]:
!git clone https://github.com/dmw1998/WASP_DL4NLP26.git 2>/dev/null || true
%cd WASP_DL4NLP26/Assignments
!ls

/workspaces/WASP_DL4NLP26/WASP_DL4NLP26/Assignments
A1  A2


In [ ]:
!pip install -q datasets nltk scikit-learn matplotlib transformers accelerate


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
import os, math, sys
import torch
import nltk

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Reuse A1's tokenizer and trainer.
sys.path.insert(0, './A1/a1_1')
sys.path.insert(0, './A2')

TRAIN_FILE = './A1/a1_1/train.txt'
VAL_FILE = './A1/a1_1/val.txt'
assert os.path.exists(TRAIN_FILE), f'{TRAIN_FILE} not found'
assert os.path.exists(VAL_FILE), f'{VAL_FILE} not found'

CUDA available: False


## Step 0: load A1 tokenizer

> **Notes**
> - Tokenizer and trainer are reused **verbatim** from A1
> - Only the model class changes (RNN → Transformer)
> - Vocab size, special tokens, training data are identical

In [ ]:
from A2.A1_skeleton import build_tokenizer, A1Tokenizer, lowercase_tokenizer, A1Trainer

MAX_VOC_SIZE = 10000
MODEL_MAX_LENGTH = 128

tokenizer = build_tokenizer(
    train_file=TRAIN_FILE,
    tokenize_fun=lowercase_tokenizer,
    max_voc_size=MAX_VOC_SIZE,
    model_max_length=MODEL_MAX_LENGTH,
)
print('vocab size:', len(tokenizer))
print('pad_token_id:', tokenizer.pad_token_id)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


vocab size: 10000
pad_token_id: 0


## Step 1: Setting up the Transformer

### Configuration

Hyperparameter overview for a small OLMo-2-style model. We use a small model (3 layers, hidden 256, 4 heads) to keep training time under 10 minutes on Colab T4.

In [ ]:
from A2_skeleton import A2ModelConfig

config = A2ModelConfig(
    vocab_size=len(tokenizer),
    hidden_size=256,
    intermediate_size=512,         # SwiGLU intermediate dim (~2x hidden)
    num_attention_heads=4,         # 256 / 4 = 64 dim per head
    num_hidden_layers=3,
    rope_theta=10000.0,            # OLMo-2 default
    hidden_act='silu',
    max_position_embeddings=MODEL_MAX_LENGTH,
    rms_norm_eps=1e-5,
)
print(f'hidden_size: {config.hidden_size}')
print(f'head_dim: {config.hidden_size // config.num_attention_heads}')
print(f'intermediate_size: {config.intermediate_size}')
print(f'num_hidden_layers: {config.num_hidden_layers}')

hidden_size: 256
head_dim: 64
intermediate_size: 512
num_hidden_layers: 3


### 🎓 Task 1.1: MLP layer (SwiGLU)

> **Notes**
> - SwiGLU = `down(SiLU(gate(x)) ⊗ up(x))` — two parallel projections, element-wise multiply, project back
> - **Three Linears** all `bias=False`: `gate_proj` (H→I), `up_proj` (H→I), `down_proj` (I→H)
> - **SiLU(x) = x · sigmoid(x)** — also called Swish; smooth ReLU-like activation
> - **Why gated?** The element-wise multiplication lets the network learn input-dependent feature selection (the `gate` branch acts as a multiplicative attention over the `up` branch)
> - **vs vanilla MLP**: `down(SiLU(up(x)))` is one matmul fewer but less expressive; SwiGLU empirically gives better perplexity per parameter

In [ ]:
from A2_skeleton import A2MLP

mlp = A2MLP(config)
x = torch.randn(2, 7, config.hidden_size)
out = mlp(x)
print(f'in:  {x.shape}')
print(f'out: {out.shape}')
assert out.shape == x.shape
print('Task 1.1 sanity check passed')

in:  torch.Size([2, 7, 256])
out: torch.Size([2, 7, 256])
Task 1.1 sanity check passed


### ⚙ Task 1.2: RMSNorm

> **Notes**
> - We use PyTorch's built-in `nn.RMSNorm` (allowed by the assignment)
> - `RMSNorm(x) = x / sqrt(mean(x²) + eps) · γ` — no centering (no mean subtraction), unlike LayerNorm
> - `elementwise_affine=True` → learnable per-channel scale γ
> - **Why RMSNorm over LayerNorm?** Simpler (one stat instead of two), ~30% faster, no quality loss in practice; standard in modern LLMs (Llama, OLMo, Qwen, etc.)
> - **Where used in OLMo-2**: after attention, after MLP, on Q and K projections inside attention, and once before the final unembedding

In [ ]:
from A2_skeleton import A2RMSNorm

norm = A2RMSNorm(config)
out = norm(x)
print(f'in:  {x.shape}')
print(f'out: {out.shape}')
assert out.shape == x.shape
print('Task 1.2 sanity check passed')

in:  torch.Size([2, 7, 256])
out: torch.Size([2, 7, 256])
Task 1.2 sanity check passed


### 🎓 Task 1.3: Multi-head attention

> **Notes — components**
> - 4 square `nn.Linear` (all `bias=False`): `q_proj`, `k_proj`, `v_proj`, `o_proj`, each `hidden_size → hidden_size`
> - OLMo-2 adds **RMSNorm on Q and K** ("QK-norm") — stabilizes training, prevents attention logit explosions
> 
> **Notes — forward, step by step**
> 1. Project: `q = q_norm(W_Q · x)`, `k = k_norm(W_K · x)`, `v = W_V · x` — shape (B, M, D)
> 2. Split heads: `view(B, M, n_h, d_h).transpose(1, 2)` → (B, n_h, M, d_h)
> 3. Apply RoPE rotations to q and k only (NOT v) — rotations encode position
> 4. `F.scaled_dot_product_attention(q, k, v, is_causal=True)` — does scaling, masking, softmax, weighted sum in one call
> 5. Merge heads: `transpose(1, 2).reshape(B, M, D)`
> 6. Output projection: `W_O · attn_out`
> 
> **Notes — why each piece**
> - **Multi-head**: lets the model attend to different subspaces simultaneously (one head might track syntax, another semantics)
> - **Scaling by √d_h**: keeps dot products from growing with d_h, preventing softmax saturation
> - **Causal mask**: position i can only attend to positions ≤ i — required for autoregressive LM
> - **RoPE**: rotates query and key vectors based on position; the dot product q·k becomes a function of *relative* position (i−j), not absolute. Better generalization to unseen lengths than learned positional embeddings.

In [ ]:
from A2_skeleton import A2Attention, A2RotaryEmbedding

attn = A2Attention(config)
rope = A2RotaryEmbedding(config)

# RoPE needs to know sequence length, which it reads from input shape[1].
dummy_ids = torch.zeros(2, 7, dtype=torch.long)
rope_rotations = rope(dummy_ids)
print(f'RoPE cos shape: {rope_rotations[0].shape}')
print(f'RoPE sin shape: {rope_rotations[1].shape}')

out = attn(x, rope_rotations)
print(f'\nattention in:  {x.shape}')
print(f'attention out: {out.shape}')
assert out.shape == x.shape
print('Task 1.3 sanity check passed')

RoPE cos shape: torch.Size([1, 7, 64])
RoPE sin shape: torch.Size([1, 7, 64])

attention in:  torch.Size([2, 7, 256])
attention out: torch.Size([2, 7, 256])
Task 1.3 sanity check passed


### 🎓 Task 1.4: Full Transformer decoder layer

> **Notes**
> - Two sublayers: **self-attention** + **MLP (SwiGLU)**
> - Each sublayer has a **residual connection**: `x = x + sublayer(x)`
> - OLMo-2 uses **post-norm** (norm AFTER sublayer, BEFORE adding residual):
>   ```
>   x = x + RMSNorm(Attention(x))
>   x = x + RMSNorm(MLP(x))
>   ```
> - **Why residual?** Gradient flows directly back through `+`, mitigates vanishing gradients in deep stacks; also lets layers learn *delta* updates rather than full transformations
> - **Post-norm vs pre-norm**: pre-norm (`x = x + Sublayer(RMSNorm(x))`, used by Llama) is easier to train; post-norm (used by OLMo-2 and original Transformer) needs more care but can give better final quality

In [ ]:
from A2_skeleton import A2DecoderLayer

layer = A2DecoderLayer(config)
out = layer(x, rope_rotations)
print(f'in:  {x.shape}')
print(f'out: {out.shape}')
assert out.shape == x.shape
print('Task 1.4 sanity check passed')

in:  torch.Size([2, 7, 256])
out: torch.Size([2, 7, 256])
Task 1.4 sanity check passed


### ⚙ Task 1.5: Complete Transformer stack

> **Notes**
> - Top-level structure: `embed_tokens` → N × `A2DecoderLayer` → final `RMSNorm` → `lm_head` (unembedding)
> - Layers stored in `nn.ModuleList` (not plain Python list) so parameters get registered for autograd
> - `lm_head` has `bias=False` (OLMo-2 convention)
> - **RoPE computed once** at the top of `forward`, then **shared** across all layers — saves recomputation
> - Loss uses **shift-by-one** identical to A1: drop last logit, drop first label

In [ ]:
from A2_skeleton import A2Transformer

model = A2Transformer(config)
n_params = sum(p.numel() for p in model.parameters())
print(f'parameters: {n_params:,}')

# Sanity check: input integer tensor → 3D logits tensor.
input_ids = torch.randint(0, len(tokenizer), (2, 7))
out = model(input_ids)
print(f'\ninput shape:  {input_ids.shape}')
print(f'logits shape: {out.logits.shape}')
expected = (2, 7, len(tokenizer))
assert out.logits.shape == expected

# Loss check.
out = model(input_ids, labels=input_ids)
print(f'loss (random init): {out.loss.item():.4f} (expect ≈ log({len(tokenizer)}) = {math.log(len(tokenizer)):.2f})')
print('Task 1.5 sanity check passed')

parameters: 7,089,408

input shape:  torch.Size([2, 7])
logits shape: torch.Size([2, 7, 10000])
loss (random init): 9.3759 (expect ≈ log(10000) = 9.21)
Task 1.5 sanity check passed


## Step 2: Training

### ⚙ Task 2.1: train the Transformer LM

> **Notes**
> - Reuse the same `A1Trainer` from Assignment 1 — interface is identical (model takes `input_ids` + `labels`, returns `.loss`)
> - Hyperparameters: small model (3 layers, hidden 256, 4 heads, ~3M params), AdamW lr=3e-4, 3 epochs, batch_size=32
> - **Lower learning rate than A1** (3e-4 vs 1e-3): Transformers are typically more sensitive to LR than RNNs
> - Expect val perplexity to be **lower than A1's RNN** (around 60-80 for this small Transformer)

In [ ]:
from datasets import load_dataset
from torch.utils.data import Subset
from transformers import TrainingArguments

dataset = load_dataset('text', data_files={'train': TRAIN_FILE, 'val': VAL_FILE})
dataset = dataset.filter(lambda x: x['text'].strip() != '')
print('train size:', len(dataset['train']))
print('val size:  ', len(dataset['val']))

dev_mode = False  # set True for a quick (~1 min) sanity run

if dev_mode:
    train_ds = Subset(dataset['train'], range(1000))
    val_ds = Subset(dataset['val'], range(200))
    epochs = 1
else:
    train_ds = dataset['train']
    val_ds = dataset['val']
    epochs = 3

Generating train split: 294118 examples [00:00, 318799.95 examples/s]
Generating val split: 35748 examples [00:00, 645097.08 examples/s]
Filter: 100%|██████████| 35748/35748 [00:00<00:00, 558397.03 examples/s]

train size: 147059
val size:   17874


In [ ]:
args = TrainingArguments(
    output_dir='trainer_output_a2',
    optim='adamw_torch',
    eval_strategy='epoch',
    learning_rate=3e-4,            # smaller than A1 — transformers need it
    num_train_epochs=epochs,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    logging_steps=100,
    save_strategy='no',
    report_to='none',
    use_cpu=False,
)

trainer = A1Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
)
trainer.train()

Device: cpu


KeyboardInterrupt: 

## Step 3: Text generation

### ⚙ Task 3.1: predict next word (greedy)

> **Notes**
> - Same as A1 5.1: take logits at position **-2** (one before `<EOS>`)
> - `argmax` → most likely next token id → look up word

In [ ]:
def predict_next(model, tokenizer, prompt, k=5):
    device = next(model.parameters()).device
    model.eval()
    enc = tokenizer(prompt, return_tensors='pt')
    input_ids = enc['input_ids'].to(device)
    with torch.no_grad():
        out = model(input_ids)
    logits = out.logits[0, -2]   # position before <EOS>
    topk = torch.topk(logits, k)
    return [(tokenizer.int_to_str[i.item()], s.item())
            for i, s in zip(topk.indices, topk.values)]

for prompt in ['She lives in San',
               'The president of the United',
               'The capital of Sweden is']:
    print(f'prompt: {prompt!r}')
    for word, score in predict_next(model, tokenizer, prompt):
        print(f'  {word:20s} {score:.3f}')
    print()

### 🎓 Task 3.2: text generation with sampling

> **Notes — algorithm**
> 1. Encode prompt → `input_ids`
> 2. Loop until `<EOS>` or `max_length`:
>    a. Forward pass → take logits at last position
>    b. Divide logits by `temperature` (higher → more random)
>    c. If `topk`: keep only top-K logits, set rest to -inf
>    d. Sample one token from the resulting distribution
>    e. Append to `input_ids`
> 
> **Notes — knobs and their effects**
> - **temperature → 0**: deterministic, picks argmax every time (= greedy decoding, can loop)
> - **temperature = 1**: sample from the model's actual distribution
> - **temperature → ∞**: uniform random (gibberish)
> - **top-K**: truncate to K most likely tokens before sampling — prevents picking very unlikely tokens
> - **Why sampling?** Greedy decoding is deterministic but often produces repetitive, bland text. Sampling adds variety. Top-K + moderate temperature is the sweet spot.
> 
> **Notes — observations to mention orally**
> - low temp + low top-K = repetitive, on-topic but bland
> - high temp + large top-K = creative but often nonsensical / ungrammatical
> - moderate (T≈0.8, K≈40) is the typical default for production LLMs

In [ ]:
from torch.distributions import Categorical

def generate(model, tokenizer, prompt, max_length=50, temperature=1.0, topk=None):
    """Sample text autoregressively from the model."""
    device = next(model.parameters()).device
    model.eval()

    # Encode prompt (don't append <EOS> — we'll generate beyond it).
    # We use the tokenizer normally and then strip the trailing EOS.
    enc = tokenizer(prompt, return_tensors='pt')
    input_ids = enc['input_ids'].to(device)
    if input_ids[0, -1].item() == tokenizer.eos_token_id:
        input_ids = input_ids[:, :-1]

    generated = []
    with torch.no_grad():
        for _ in range(max_length):
            out = model(input_ids)
            logits = out.logits[0, -1]                       # last position
            logits = logits / max(temperature, 1e-8)         # temperature scaling

            if topk is not None:
                top_vals, top_idx = torch.topk(logits, topk)
                # Mask everything outside top-K to -inf so softmax ignores it.
                mask = torch.full_like(logits, float('-inf'))
                mask[top_idx] = top_vals
                logits = mask

            dist = Categorical(logits=logits)
            next_id = dist.sample()
            if next_id.item() == tokenizer.eos_token_id:
                break
            generated.append(next_id.item())
            input_ids = torch.cat([input_ids, next_id.view(1, 1)], dim=1)

    # Decode: prompt + sampled words.
    prompt_words = tokenizer.decode(enc['input_ids'][0]) if hasattr(tokenizer, 'decode') \
        else [tokenizer.int_to_str[i.item()] for i in enc['input_ids'][0]
              if i.item() not in (tokenizer.pad_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id)]
    gen_words = [tokenizer.int_to_str[i] for i in generated]
    return ' '.join(prompt_words + gen_words)

In [ ]:
# Run with several prompts and parameter combinations.
prompts = [
    'In natural language processing , a transformer',
    'Is stockholm the capital of sweden ? answer yes or no . the answer is',
    'Write a python program that reverses a list .',
]

configs = [
    {'temperature': 0.3, 'topk': 5,    'label': 'low temp / low K (conservative)'},
    {'temperature': 0.8, 'topk': 40,   'label': 'moderate (LLM default)'},
    {'temperature': 1.5, 'topk': None, 'label': 'high temp / no K (chaotic)'},
]

torch.manual_seed(17)
for prompt in prompts:
    print(f'=== PROMPT: {prompt!r} ===')
    for cfg in configs:
        label = cfg.pop('label')
        text = generate(model, tokenizer, prompt, max_length=40, **cfg)
        print(f'\n[{label}]')
        print(text)
        cfg['label'] = label   # restore for next prompt
    print('\n' + '='*70 + '\n')

### 🎓 Task 3.3: Compare to pre-trained OLMo-2-1B

> **Notes**
> - Download ~4GB of weights — slow on first run
> - OLMo-2-1B was trained on **trillions** of tokens vs our ~150k Wikipedia lines
> - It's a **base model** (not instruction-tuned) — doesn't follow instructions, just continues text
> - Expected differences from our model:
>   - Fluent, coherent multi-sentence text
>   - Real factual knowledge ("capital of Sweden" → "Stockholm")
>   - Wider vocabulary (BPE subwords, not whole words)
>   - Can almost write valid Python
>   - Still hallucinates and may repeat
> 
> **Notes — why the gap?**
> - **Scale of data**: 4-5 orders of magnitude more text
> - **Scale of model**: 1B params vs our ~3M (~300x)
> - **Tokenizer**: subword BPE vs whole-word; OLMo-2's vocab is 50k while ours is 10k
> - **Context length**: 4096 vs our 128

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = 'allenai/OLMo-2-0425-1B'
olmo_tok = AutoTokenizer.from_pretrained(model_name)
olmo = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
olmo = olmo.to('cuda' if torch.cuda.is_available() else 'cpu')
olmo.eval()

print(f'OLMo-2-1B params: {sum(p.numel() for p in olmo.parameters()):,}')

In [ ]:
def generate_olmo(prompt, max_new_tokens=60, temperature=0.8, top_k=40):
    inputs = olmo_tok(prompt, return_tensors='pt').to(olmo.device)
    with torch.no_grad():
        out_ids = olmo.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            pad_token_id=olmo_tok.eos_token_id,
        )
    return olmo_tok.decode(out_ids[0], skip_special_tokens=True)

torch.manual_seed(17)
olmo_prompts = [
    'In natural language processing, a Transformer',
    'Is Stockholm the capital of Sweden? Answer yes or no. The answer is',
    'Write a Python program that reverses a list.',
]
for p in olmo_prompts:
    print(f'=== {p!r} ===')
    print(generate_olmo(p))
    print()

## Wrap-up: oral exam quick-reference

**Most likely follow-ups:**

- **Why RoPE over learned position embeddings?** Generalizes to longer sequences; encodes *relative* position naturally via the q·k dot product
- **Why causal mask?** Without it, position *i* could attend to future tokens — defeats autoregressive LM
- **Why scale by √d_h?** Dot product of two random d_h-dim vectors has variance d_h; scaling keeps softmax in a reasonable range
- **Why multi-head?** Single softmax can only attend to one thing at a time; multiple heads let the model track multiple types of relations in parallel
- **Why pre-norm vs post-norm?** Pre-norm trains more stably; post-norm (OLMo-2, original) needs warmup but can be slightly better at convergence
- **Why SwiGLU over plain MLP?** Gating mechanism lets the network learn input-dependent feature selection; empirically lower perplexity per parameter
- **Why bias=False everywhere?** OLMo-2 convention; saves params; RMSNorm provides the affine offset that biases would otherwise give
- **Transformer vs RNN trade-offs?** Transformer: parallelizable across positions (fast training), but O(n²) attention (slow for long contexts). RNN: O(n) but sequential (slow training), forgets long-range info.